# Task 2 — Incremental CPG parser

```{admonition} Evidence summary
:class: note

| Field | Value |
|---|---|
| Evidence source | `screenshots/stage2_manifest.json` |
| Command | `docker compose run --rm ... parser python -m parser_service.main --repo data/datasets --mode full` |
| Parser scope | Every discovered core Python file, processed one file at a time. |
| Stable identity | Repository, commit, file identity, node location/type, and edge endpoints. |
```

## Approach and reasoning

Command: `docker compose run --rm ... parser python -m parser_service.main --repo data/datasets --mode full`. The parser emits semantic AST nodes, synthetic per-function exit nodes, CFG, DFG, and CALL records with stable IDs. Every internal edge endpoint is audited against the emitted node set, and files are processed and flushed one at a time.

## What this chapter proves

| Requirement | Evidence in this chapter |
|---|---|
| Incremental CPG extraction | Executed output reports node and edge counts from the captured run. |
| Error handling | The run preserves the intentional syntax-error event instead of dropping it. |
| Deterministic IDs | The text explains the ID ingredients used by the parser and graph sinks. |


In [1]:
from pathlib import Path
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'screenshots/stage2_manifest.json').exists())
manifest = json.loads((ROOT / 'screenshots/stage2_manifest.json').read_text())
m = manifest['metrics']['kafka']
print('AST nodes:', m['ast_nodes'])
print('CFG edges:', m['cfg_edges'])
print('DFG edges:', m['dfg_edges'])
print('CALL edges:', m['call_edges'])
print('total edges:', m['total_edges'])
print('metadata events:', m['metadata_events'], 'error events:', m['error_events'])


AST nodes: 133263
CFG edges: 18097
DFG edges: 8586
CALL edges: 11458
total edges: 38141
metadata events: 138 error events: 1


## Reflection

```{admonition} Closing reflection
:class: tip

| Point | Result |
|---|---|
| Worked | Every discovered core source file produced AST, CFG, DFG, and CALL structures; a separate controlled file produced the parser-error event. |
| Failed | A previous capture sampled fewer metadata/error records than the assignment requires. |
| Resolution | The canonical run now uses `--mode full`, requires one metadata event and one MongoDB document per discovered file, and rejects partial-repository evidence. |
```
